# Creating the HTLC-Timeout Transaction

In this section, we'll build a Lightning channel HTLC-timeout transaction v3 from scratch using Python. We'll walk through each part of the transaction — how it's constructed and signed. The process will be tested using Bitcoin Core in regtest mode.

## Setup

For this notebook, we'll use the commitment transaction created in `chapter 3 - in-flight htlc commitment transaction v3` which contains the offered HTLC output that we'll spend with the HTLC-timeout transaction.

In [1]:
%run "../chapter 3 - in-flight htlc commitment transaction/in-flight htlc commitment transaction v3.ipynb"

2026-04-13T19:13:30.217000Z TestFramework (INFO): PRNG seed is: 7048875519951549855
2026-04-13T19:13:30.220000Z TestFramework (INFO): Initializing test directory /tmp/bitcoin_func_test_u3ihrgn1
🟢 New TestShell started. Block height: 0
Alice Per Commitment Seed 34b581ec20bf2c6cae3d4d4dcbfddc8a3727a1e9a57c55f3520e770607898c06
Bob Per Commitment Seed 89c994b3ddad4698acee71e42d8bcace48eea739caaba371eb110e77663ec56d
Alice Payment BasePoint:  025f892a06124391e2f38ce35d943cdc09f63e203330dbd9cb6113a903e0738458
Bob Payment BasePoint:  02f98efd3f2b2fbe7bd83c419f5f64f8280798b8a9175fdb77c0091bbb95c79506
To obscure commitment number 0xb433fd43a66f
Alice funding pubkey: d7a585dd66c254eba42231220e1687e355397d17d6c15b37c32dbf08bbc69000
Alice funding privkey: 06691bb51294cb8afd1d7bdb05194a4daaee7b9546b166574e3d7551d7439b82
Alice funding address: bcrt1p67jcthtxcf2whfpzxy3qu958ud2njlgh6mq4kd7r9kls3w7xjqqqrdvcdp
Alice sweeper pubkey: 9b4e37064b1af5e833d44fe6e1d48855b8b653cc424955594908912f4a68e78f
Alice s

## HTLC-Timeout Transaction

The HTLC-timeout transaction is a second-level transaction that spends an offered HTLC output from a commitment transaction. It allows the local party (Alice, who offered the HTLC) to reclaim the funds after the timeout period if the remote party (Bob) doesn't claim the HTLC with the preimage.

### The Unsigned HTLC-Timeout Transaction

#### The Input

The input is the offered HTLC output from Alice's commitment transaction (output index 3).

In [2]:
# Get the commitment transaction details
# Alice's second commitment transaction has the offered HTLC output at index 3
commitment_txid = decoded_alice_commitment_tx['txid']
commitment_output_index = 2

print(f"Spending from commitment tx: {commitment_txid}")

# VERSION
# version '3' indicates that we are using V3 TRUC (Topologically Restricted Until Confirmation) transactions.
version = bytes.fromhex("0300 0000")

# MARKER (new to segwit)
marker = bytes.fromhex("00")

# FLAG (new to segwit)
flag = bytes.fromhex("01")

# INPUTS
# We have just 1 input
input_count = bytes.fromhex("01")

# Convert txid and index to bytes (little endian)
commitment_txid = bytes.fromhex(commitment_txid)[::-1]
commitment_index = commitment_output_index.to_bytes(4, byteorder="little", signed=False)

# For the unsigned transaction we use an empty scriptSig
scriptsig = bytes.fromhex("")

# Sequence: must be set to enforce relative locktime (though scriptA doesn't have CSV)
# Set to 1 for option_anchors
sequence_htlc = bytes.fromhex("01000000")

inputs = (
    commitment_txid
    + commitment_index
    + varint_len(scriptsig)
    + scriptsig
    + sequence_htlc
)

Spending from commitment tx: b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049


#### The Output

The HTLC-timeout transaction has a single output that pays to Alice with a delay. This output is similar to the `to_local` output in commitment transactions: it's spendable by Alice after a delay, or immediately by Bob if he has the revocation key (in case this commitment gets revoked).

    +------+---------------+
    | OP_1 |       Q       |
    +------+---------------+
                   ^
                   |   +-------------------+
                    ---| P(revocation) + T |
                       +-------------------+
                                         ^
                                         |
                                   +-----------+        
                                   | T = t * G |
                                   +-----------+        
                                         ^
                                         |
     +---+   +-------------------------------------------------------+
     | t | = | TaggedHash ("Taptweak", P(revocation) || script_root) |
     +---+   +-------------------------------------------------------+
                                                             ^
                                                             |
                                                          +-----+
                                                          |  h  |
                                                          +-----+                   
                                                             ^                                                          
                                                             |                                                          
      +-----------------------------------------------------------+                                           
      | P(local_delayed) OP_CHECKSIG to_self_delay OP_CSV OP_DROP |                                           
      +-----------------------------------------------------------+

Obs: If option_anchors applies, which is the case here, then the HTLC-timeout and HTLC-success transactions are signed with the input and output having the same value. This means they have a zero fee and MUST be combined with other inputs to arrive at a reasonable fee.

In [3]:
# Output count
output_count = bytes.fromhex("01")

# Output value
output_value_sat = htlc_output_value_satoshis
output_value = output_value_sat.to_bytes(8, byteorder="little", signed=False)

# We already have alice_delayed_pubkey and bob_revocation_pubkey from chapter 3
# These are the same keys used in the commitment transaction to_local output

# Create the script: P(local_delayed) OP_CHECKSIG to_self_delay OP_CSV OP_DROP
script_output = CScript([alice_delayed_pubkey.get_bytes(bip340=True), OP_CHECKSIG, to_self_delay, OP_CHECKSEQUENCEVERIFY, OP_DROP])

# Compute output script_root
hash_input = TAPSCRIPT_VER + ser_string(script_output)
script_root = tagged_hash("TapLeaf", hash_input)

# Compute the output Tagged Hash
taptweak = tagged_hash("TapTweak", bob_revocation_pubkey.get_bytes(bip340=True) + script_root)
htlc_bob_revocation_pubkey_tweaked = bob_revocation_pubkey.tweak_add(taptweak)
# Compute scriptPubKey P2TR: OP_1 (0x51) + PUSH32 (0x20) + xonly(32B)
output_spk = bytes.fromhex("51") + varint_len(htlc_bob_revocation_pubkey_tweaked.get_bytes(bip340=True)) + htlc_bob_revocation_pubkey_tweaked.get_bytes(bip340=True)

outputs = (
    output_value
    + varint_len(output_spk)
    + output_spk
)

# Locktime: set to current block height + time lock delta
current_block_height = node.getblockcount()
locktime = current_block_height + 60  # assuming a 60-block of time lock delta
locktime = locktime.to_bytes(4, 'little')

unsigned_tx = (
    version
    + input_count
    + inputs
    + output_count
    + outputs
    + locktime
)
print("\nunsigned_tx:", unsigned_tx.hex())

# Decode the unsigned transaction to verify it looks correct
decoded_htlc_timeout = node.decoderawtransaction(unsigned_tx.hex())
print(json.dumps(decoded_htlc_timeout, indent=2, default=str))


unsigned_tx: 030000000149b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b80200000000010000000120a10700000000002251201bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d8a3000000
{
  "txid": "be7d77a2c0b807de9e8b3268a2d9af46fac75880e9b6bc953b8ea804dd4a35d3",
  "hash": "be7d77a2c0b807de9e8b3268a2d9af46fac75880e9b6bc953b8ea804dd4a35d3",
  "version": 3,
  "size": 94,
  "vsize": 94,
  "weight": 376,
  "locktime": 163,
  "vin": [
    {
      "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
      "vout": 2,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 1
    }
  ],
  "vout": [
    {
      "value": "0.00500000",
      "n": 0,
      "scriptPubKey": {
        "asm": "1 1bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d8",
        "desc": "rawtr(1bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d8)#afwmkgn2",
        "hex": "51201bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6c

## The sighash for script path spend

Unlike the commitment transaction which uses **key path spending**, the HTLC-timeout transaction uses **script path spending** through the Taproot tree. We need to spend using scriptA of the offered HTLC output.

The sighash calculation for script path spending is similar to key path, but includes additional data about the script being executed.

In [4]:
# We're spending the offered HTLC output using scriptA
# Recall scriptA from chapter 3: P(local_htlc) OP_CHECKSIGVERIFY P(remote_htlc) OP_CHECKSIG
spending_script = CScript([alice_htlc_pubkey.get_bytes(bip340=True), OP_CHECKSIGVERIFY, bob_htlc_pubkey.get_bytes(bip340=True), OP_CHECKSIG])

# Calculate the tapleaf hash for the script we're spending
hash_input_spending = TAPSCRIPT_VER + ser_string(spending_script)
tapleaf_hash = tagged_hash("TapLeaf", hash_input_spending)
key_version = bytes.fromhex("00")  # reserved for future ugrades
codeseparator = bytes.fromhex("ffffffff")  

# SIGHASH for script path spend (BIP-341)
index_of_this_input = bytes.fromhex("0000 0000")
sighash_epoch = bytes.fromhex("00")
hash_type = bytes.fromhex("83")  # SIGHASH_SINGLE|SIGHASH_ANYONECANPAY (0x03 | 0x80)

# When SIGHASH_SINGLE (0x03) is set, these fields are 35 bytes (BIP-341)
# corresponding to the output (32): the SHA256 of the corresponding output in CTxOut format.
sha_outputs = sha256(outputs).digest()

# Data about this input, script path (no annex)
spend_type = bytes.fromhex("02")

# Common signature message extension (BIP-341)
# When SIGHASH_ANYONECANPAY is set, we include data about THIS specific input:
outpoint = commitment_txid + commitment_index  # 36 bytes
amount = htlc_output_value  # 8 bytes
nSequence = sequence_htlc  # 4 bytes

sig_msg = (
    sighash_epoch
    + hash_type
    + version
    + locktime
    + spend_type
    + outpoint          
    + amount
    + varint_len(alice_offered_htlc_spk)
    + alice_offered_htlc_spk
    + nSequence
    + sha_outputs
    + tapleaf_hash
    + key_version
    + codeseparator
)

print("\nSignature message:", sig_msg.hex())

tag_hash = sha256("TapSighash".encode()).digest()
sighash = sha256(tag_hash + tag_hash + sig_msg).digest()
print("\nSighash:", sighash.hex())


Signature message: 008303000000a30000000249b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b80200000020a1070000000000225120d25da52cc19b2451d6162d1130f087658e6a2c388ea23c4a84bbb0451a01c4c7010000006dbca07ba4b0024cf7b46831be6cfac3f42a43e7d11851f1814e7f2719fb9d90509fbe809f2233d8daa3bca96c2735a7f7a1bd75eb48976978029ac95cd0fd1300ffffffff

Sighash: 7efd28766885036e9a17a829f58dc4f19c0c7757d7a45d08f358f0ac9c56849c


## Signing the sighash

For the HTLC-timeout transaction, we need signatures from both Alice's and Bob's HTLC keys. These are regular Schnorr signatures (not MuSig2 aggregated signatures), since the script requires two separate signature checks.

In [5]:
# Derive Alice's HTLC private key
alice_htlc_basepoint = derivate_key(alice_node_seed, family=2, channel_index=0)
alice_htlc_privkey = alice_htlc_basepoint.get_privkey(alice_per_commitment.get_pub())

# Derive Bob's HTLC private key
bob_htlc_basepoint = derivate_key(bob_node_seed, family=2, channel_index=0)
bob_htlc_privkey = bob_htlc_basepoint.get_privkey(bob_per_commitment.get_pub())

# Generate auxiliary random data for each signature
import secrets
aux_alice = secrets.token_bytes(32)
aux_bob = secrets.token_bytes(32)

# Sign with Alice's HTLC key
alice_htlc_sig = alice_htlc_privkey.sign_schnorr(sighash, aux_alice)
# APPEND sighash type byte for non-default sighash!
alice_htlc_sig = alice_htlc_sig + hash_type  # Add 0x83 byte

# Verify Alice's signature (without the sighash byte for verification)
alice_sig_valid = alice_htlc_pubkey.verify_schnorr(alice_htlc_sig[:-1], sighash)
print("Alice signature valid?", alice_sig_valid)
print(f"Alice signature length: {len(alice_htlc_sig)} bytes (should be 65)")

# Sign with Bob's HTLC key  
bob_htlc_sig = bob_htlc_privkey.sign_schnorr(sighash, aux_bob)
# APPEND sighash type byte for non-default sighash!
bob_htlc_sig = bob_htlc_sig + hash_type  # Add 0x83 byte

# Verify Bob's signature (without the sighash byte for verification)
bob_sig_valid = bob_htlc_pubkey.verify_schnorr(bob_htlc_sig[:-1], sighash)
print("Bob signature valid?", bob_sig_valid)
print(f"Bob signature length: {len(bob_htlc_sig)} bytes (should be 65)")

Alice signature valid? True
Alice signature length: 65 bytes (should be 65)
Bob signature valid? True
Bob signature length: 65 bytes (should be 65)


## The signed transaction

Now we construct the witness for the script path spend. The witness stack for spending scriptA contains:
1. Bob's signature (last signature on stack, for OP_CHECKSIG)
2. Alice's signature (for OP_CHECKSIGVERIFY)
3. The script being executed (scriptA)
4. The control block (proving scriptA is in the taproot tree)

In [6]:
# Construct the control block
# Control block format: <version byte> <internal key> <parity bit> [<merkle proof>]
# Version byte: 0xc0 | (parity of Q)

# Get parity bit from Q
parity = 0 if bob_revocation_pubkey_tweaked.get_bytes(bip340=False)[0] == 0x02 else 1
version_byte = bytes([0xc0 | parity])

control_block = version_byte + bob_revocation_pubkey.get_bytes(bip340=True) + htlc_taggedhash_leafB

print("\ncontrol_block:", control_block.hex())
print("\nspending_script:", spending_script.hex())

# Construct witness stack
# Stack order (bottom to top): <alice_sig> <bob_sig> <script> <control_block>
witness = (
    bytes.fromhex("04")  # 4 stack items
    + varint_len(bob_htlc_sig)    # Bob's signature (for OP_CHECKSIG)
    + bob_htlc_sig
    + varint_len(alice_htlc_sig)  # Alice's signature (for OP_CHECKSIGVERIFY)
    + alice_htlc_sig
    + varint_len(spending_script)  # The script
    + spending_script
    + varint_len(control_block)    # The control block
    + control_block
)

# The final signed transaction
signed_tx = (
    version
    + marker
    + flag
    + input_count
    + inputs
    + output_count
    + outputs
    + witness
    + locktime
)

print("\nsigned tx:", signed_tx.hex())



control_block: c0c39a6b0cffa569f243cec8252a1e5f93b4b072247ff18773917b72fb6341f6082fbd0ee9ad97225e09e436e3e9965d3b1bea9c935aa04af360d50ae3f8740231

spending_script: 207546c6ab3a8c005ce24e9feb7154fd35b62ac0f9125beb5e692d4d912a95de77ad207bd8383df06640c0efe760bb2bac1a1585bdc664e6f1cb61c41a76a6f2be3484ac

signed tx: 0300000000010149b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b80200000000010000000120a10700000000002251201bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d80441212f8c962727d8459d490a94310dd61ad215a66c0f9a8f29ec68869b0cf03bb24228df68388bae823738701b0a682b87718b82394b930382bbe05aed4618304f834192de7253267462a04474c872d60fcb4d3e6ef5bb596dca665792f79d89cb4dae1da7fc1a9e92b4fc5256afc7b813dda786953f210be6f638ffd638db949418b08344207546c6ab3a8c005ce24e9feb7154fd35b62ac0f9125beb5e692d4d912a95de77ad207bd8383df06640c0efe760bb2bac1a1585bdc664e6f1cb61c41a76a6f2be3484ac41c0c39a6b0cffa569f243cec8252a1e5f93b4b072247ff18773917b72fb6341f6082fbd0ee9ad97225e09e436e3e9965d

## Confirming the commitment transaction

The v3 commitment transaction must be broadcast as a package with a child transaction paying the fees.

In [7]:
# INPUTS
# We have 2 inputs: the commitment transaction shared anchor and alice sweeper output 
input_count_v2 = bytes.fromhex("02")

# INPUT 1: shared anchor output
shared_anchor_output_index = 0
shared_anchor_output_index = shared_anchor_output_index.to_bytes(4, byteorder="little", signed=False)

# INPUT 2: alice sweeper output
# Get funding transaction details from chapter 1
alice_txid_to_spend_bytes = bytes.fromhex(txid_to_spend)[::-1]
sweeper_index_bytes = (alice_sweeper_index).to_bytes(4, byteorder="little", signed=False)
sequence = bytes.fromhex("ffffffff")

inputs_v2 = (
    commitment_txid
    + shared_anchor_output_index
    + varint_len(scriptsig)
    + scriptsig
    + sequence
    + alice_txid_to_spend_bytes
    + sweeper_index_bytes
    + varint_len(scriptsig)
    + scriptsig
    + sequence
)

# OUTPUTS
# change back to alice
output_count_v2 = bytes.fromhex("01")

# OUTPUT : Change back to alice_change address
# The 240 sats from shared_anchor will pay for the fees + 500 sats from alice_sweeper output.
alice_change_value = int(sweeper_initial_fund * 100000000) - 500 
alice_change_value = alice_change_value.to_bytes(8, byteorder="little", signed=False)

# scriptPubKey P2TR: OP_1 (0x51) + PUSH32 (0x20) + alice_change_pubkey
output_spk_change = bytes.fromhex("51") + varint_len(alice_change_pubkey.get_bytes(bip340=True)) + alice_change_pubkey.get_bytes(bip340=True)

# LOCKTIME for the child (commitment cpfp) transaction — use a distinct name so we
# don't clobber the HTLC-timeout locktime that was set in the unsigned_tx cell.
child_locktime = bytes.fromhex("00000000")

# Use outputs_v2 to avoid overwriting the original outputs variable
outputs_v2 = (
    alice_change_value
    + varint_len(output_spk_change)
    + output_spk_change
)

unsigned_tx_v2 = (
    version
    + input_count_v2
    + inputs_v2
    + output_count_v2
    + outputs_v2
    + child_locktime
)

print("unsigned_tx_v2:", unsigned_tx_v2.hex())

# Decode the unsigned transaction to verify it looks correct
decoded_child_tx = node.decoderawtransaction(unsigned_tx_v2.hex())
print(json.dumps(decoded_child_tx, indent=2, default=str))

unsigned_tx_v2: 030000000249b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b80000000000ffffffff7c85ae64a3a7a0438e39b3078f01b2ef45aee989258d08e053cc1eb8858b29840100000000ffffffff018c949800000000002251208560a3ef21475305df5fdf049b299f5db8efc55668867ae1212061ee7e901c1c00000000
{
  "txid": "6367ae021474c85c64305a35749aa4de39d63cf9fefbcf429f3d2b2801d7d886",
  "hash": "6367ae021474c85c64305a35749aa4de39d63cf9fefbcf429f3d2b2801d7d886",
  "version": 3,
  "size": 135,
  "vsize": 135,
  "weight": 540,
  "locktime": 0,
  "vin": [
    {
      "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
      "vout": 0,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 4294967295
    },
    {
      "txid": "84298b85b81ecc53e0088d2589e9ae45efb2018f07b3398e43a0a7a364ae857c",
      "vout": 1,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 4294967295
    }
  ],
  "vout": [
    {
      "value": "0.0999950

### The sighash for the key path spend

This is the “Schnorr key spend” the message preimage is called msg_digest in [BIP-341](https://github.com/bitcoin/bips/blob/master/bip-0341.mediawiki).

In [8]:
# SIGHASH for the shared anchor input (key path spend)
index_of_this_input = bytes.fromhex("0100 0000")

alice_sweeper_value = int(sweeper_initial_fund * 100000000)

# SIGHASH for key path spend
# See BIP-341 for details
sighash_epoch = bytes.fromhex("00")

# Control
hash_type = bytes.fromhex("00") # SIGHASH_DEFAULT (a new sighash type meaning implied SIGHASH_ALL)

# Transaction data
sha_prevouts = sha256(commitment_txid + shared_anchor_output_index + alice_txid_to_spend_bytes + sweeper_index_bytes).digest()
sha_amounts = sha256(anchor_output_value + alice_sweeper_value.to_bytes(8, byteorder="little", signed=False)).digest()

# scriptPubKeys for both inputs
shared_anchor_spk_with_len = varint_len(shared_anchor_spk) + shared_anchor_spk
alice_sweeper_spk = bytes.fromhex("51") + varint_len(alice_sweeper_pubkey.get_bytes(bip340=True)) + alice_sweeper_pubkey.get_bytes(bip340=True)
alice_sweeper_spk_with_len = varint_len(alice_sweeper_spk) + alice_sweeper_spk
sha_scriptpubkeys = sha256(shared_anchor_spk_with_len + alice_sweeper_spk_with_len).digest()

sha_sequences = sha256(sequence+sequence).digest()

sha_outputs = sha256(outputs_v2).digest()

# Data about this input
spend_type = bytes.fromhex("00") # no annex present

sig_msg = (
    sighash_epoch
    + hash_type
    + version
    + child_locktime
    + sha_prevouts
    + sha_amounts
    + sha_scriptpubkeys
    + sha_sequences
    + sha_outputs
    + spend_type
    + index_of_this_input
)

tag_hash = sha256("TapSighash".encode()).digest()
msg = tag_hash + tag_hash + sig_msg
sighash = sha256(msg).digest()

# Sign with Alice's sweeper private key.
# SIGHASH_DEFAULT → 64-byte signature, no hash-type byte appended.
aux_sweeper = secrets.token_bytes(32)
alice_sweeper_sig = alice_sweeper_privkey.sign_schnorr(sighash, aux_sweeper)

alice_sweeper_sig_valid = alice_sweeper_pubkey.verify_schnorr(alice_sweeper_sig, sighash)
print("Alice sweeper signature valid?", alice_sweeper_sig_valid)
print(f"Alice sweeper sig ({len(alice_sweeper_sig)} bytes):", alice_sweeper_sig.hex())


Alice sweeper signature valid? True
Alice sweeper sig (64 bytes): a29cc17a5d566621fb6c91b02ab486abc857ac056d227cd73b8f28022f0cc659658bd3367a330319e022df973976db43475f44e3eb8de052195be17d817d5fcd


### Building the witness and signed child transaction

Two inputs, two witnesses:
- **Input 0** (shared_anchor `OP_1 <0x4e73>`): P2A — **empty witness**. This is an anyone-can-spend output; no key or signature is needed.
- **Input 1** (Alice's sweeper P2TR): **Schnorr signature** from `alice_sweeper_privkey` (key path spend, SIGHASH_DEFAULT → 64-byte sig, no hash-type byte appended).

In [9]:
# Build witnesses for each input:
# - Input 0 (shared_anchor): P2A output (OP_1 <0x4e73>) — anyone-can-spend, empty witness
# - Input 1 (Alice's sweeper): P2TR key path — Schnorr signature from Alice's sweeper key

witness_p2a = bytes.fromhex("00")   # 0 stack items — no signature required for P2A

witness_sweeper = (
    bytes.fromhex("01")              # 1 stack item
    + varint_len(alice_sweeper_sig)  # 64 bytes (SIGHASH_DEFAULT)
    + alice_sweeper_sig
)

# Construct the signed child transaction
signed_child_tx = (
    version
    + marker
    + flag
    + input_count_v2
    + inputs_v2
    + output_count_v2
    + outputs_v2
    + witness_p2a
    + witness_sweeper
    + child_locktime
)

print("Signed child tx:", signed_child_tx.hex())
decoded_signed_child = node.decoderawtransaction(signed_child_tx.hex())
print(json.dumps(decoded_signed_child, indent=2, default=str))

Signed child tx: 0300000000010249b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b80000000000ffffffff7c85ae64a3a7a0438e39b3078f01b2ef45aee989258d08e053cc1eb8858b29840100000000ffffffff018c949800000000002251208560a3ef21475305df5fdf049b299f5db8efc55668867ae1212061ee7e901c1c000140a29cc17a5d566621fb6c91b02ab486abc857ac056d227cd73b8f28022f0cc659658bd3367a330319e022df973976db43475f44e3eb8de052195be17d817d5fcd00000000
{
  "txid": "6367ae021474c85c64305a35749aa4de39d63cf9fefbcf429f3d2b2801d7d886",
  "hash": "048bebc5275c1a92610aae6918052996cf469e1d741dcb6ce2ef16271165ec17",
  "version": 3,
  "size": 204,
  "vsize": 153,
  "weight": 609,
  "locktime": 0,
  "vin": [
    {
      "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
      "vout": 0,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 4294967295
    },
    {
      "txid": "84298b85b81ecc53e0088d2589e9ae45efb2018f07b3398e43a0a7a364ae857c",
      "vout": 1,
      "scri

### Broadcasting as a v3 package

Since the commitment transaction is v3 (TRUC), it must be broadcast together with the child that pays its fees. Bitcoin Core's `submitpackage` RPC handles this atomically — both transactions enter the mempool together or neither does.

In [10]:
# Submit the commitment + child as a v3 (TRUC) package.
# The child (which spends the P2A anchor + Alice's UTXO) pays fees for the whole package.

print("Testing package acceptance...")
result = node.testmempoolaccept([signed_alice_commitment_tx.hex(), signed_child_tx.hex()])
print(json.dumps(result, indent=2, default=str))

print("\nSubmitting package...")
result = node.submitpackage([signed_alice_commitment_tx.hex(), signed_child_tx.hex()])
print(json.dumps(result, indent=2, default=str))

# We mine enough blocks to satisfy the htlc time lock delta.
result = node.generatetoaddress(nblocks=60, address=address, called_by_framework=True)
print(json.dumps(result, indent=2, default=str))

Testing package acceptance...
[
  {
    "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
    "wtxid": "6178173f95c3051cbacfbf13305c8e8f8fa830f0e91ac09bd88e637e42bf23d5",
    "allowed": false,
    "reject-reason": "min relay fee not met",
    "reject-details": "min relay fee not met, 0 < 167"
  },
  {
    "txid": "6367ae021474c85c64305a35749aa4de39d63cf9fefbcf429f3d2b2801d7d886",
    "wtxid": "048bebc5275c1a92610aae6918052996cf469e1d741dcb6ce2ef16271165ec17"
  }
]

Submitting package...
{
  "package_msg": "success",
  "tx-results": {
    "6178173f95c3051cbacfbf13305c8e8f8fa830f0e91ac09bd88e637e42bf23d5": {
      "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
      "vsize": 167,
      "fees": {
        "base": "0E-8",
        "effective-feerate": "0.00002312",
        "effective-includes": [
          "6178173f95c3051cbacfbf13305c8e8f8fa830f0e91ac09bd88e637e42bf23d5",
          "048bebc5275c1a92610aae6918052996cf469e1d741dcb6ce2ef16

## Adding an input to the HTLC-timeout transaction to pay for fees

As stated in the BOLTs: "If option_anchors applies, then the HTLC-timeout and HTLC-success transactions are signed with the input and output having the same value. This means they have a zero fee and MUST be combined with additional inputs to reach a reasonable fee."

We now modify the transaction by adding Alice’s change output (created above) as a second input to pay for fees.

In [11]:
# INPUTS
# We have 2 inputs: the HTLC output and alice change output 
input_count = bytes.fromhex("02")

# INPUT 2: alice change output from the confirmed child tx
# decoded_signed_child['txid'] returns a hex string — must convert to bytes (little-endian)
change_txid = bytes.fromhex(decoded_signed_child['txid'])[::-1]
change_index = 0
change_index_bytes = change_index.to_bytes(4, byteorder="little", signed=False)
sequence_change = bytes.fromhex("ffffffff")

inputs_v2 = (
    commitment_txid
    + commitment_index
    + varint_len(scriptsig)
    + scriptsig
    + sequence_htlc
    + change_txid
    + change_index_bytes
    + varint_len(scriptsig)
    + scriptsig
    + sequence_change
)

# OUTPUTS
# Now we have 2 outputs: HTLC delayed output + change back to alice
output_count_v2 = bytes.fromhex("02")

# OUTPUT 1: HTLC delayed output (same as before)
# Reuse the same output_value and output_spk from v1

# OUTPUT 2: Change back to alice_change address
# alice_change_value is already bytes at this point — recompute the sat amount from scratch
# Note: the child tx already consumed 500 sats from alice's sweeper UTXO as fees,
# so the actual input here is (sweeper_initial_fund * 1e8 - 500) sats.
tx_fee_sat = 300
alice_change_value_sat = int(sweeper_initial_fund * 100000000) - 500 - tx_fee_sat
alice_change_value = alice_change_value_sat.to_bytes(8, byteorder="little", signed=False)

# scriptPubKey P2TR: OP_1 (0x51) + PUSH32 (0x20) + alice_change_pubkey
output_spk_change = bytes.fromhex("51") + varint_len(alice_change_pubkey.get_bytes(bip340=True)) + alice_change_pubkey.get_bytes(bip340=True)

outputs_v2 = (
    output_value
    + varint_len(output_spk)
    + output_spk
    + alice_change_value
    + varint_len(output_spk_change)
    + output_spk_change
)

unsigned_tx_v2 = (
    version
    + input_count
    + inputs_v2
    + output_count_v2
    + outputs_v2
    + locktime
)

print("unsigned_tx_v2:", unsigned_tx_v2.hex())

decoded_htlc_timeout_v2 = node.decoderawtransaction(unsigned_tx_v2.hex())
print(json.dumps(decoded_htlc_timeout_v2, indent=2, default=str))

unsigned_tx_v2: 030000000249b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b802000000000100000086d8d701282b3d9f42cffbfef93cd639dea49a74355a30645cc8741402ae67630000000000ffffffff0220a10700000000002251201bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d860939800000000002251208560a3ef21475305df5fdf049b299f5db8efc55668867ae1212061ee7e901c1ca3000000
{
  "txid": "9f16971f6a5f723e89a2935a3e7f2fec0dfd2976f271018783e7f2186b6d7166",
  "hash": "9f16971f6a5f723e89a2935a3e7f2fec0dfd2976f271018783e7f2186b6d7166",
  "version": 3,
  "size": 178,
  "vsize": 178,
  "weight": 712,
  "locktime": 163,
  "vin": [
    {
      "txid": "b8c1fc1fb3d04182879e82a300237d280d68a3f7d4c25bbdf93092975902b049",
      "vout": 2,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 1
    },
    {
      "txid": "6367ae021474c85c64305a35749aa4de39d63cf9fefbcf429f3d2b2801d7d886",
      "vout": 0,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
    

### Sign the alice sweeper input (Input 1 - key path spend)

The HTLC input signatures were already created earlier and can be reused. We only need to sign the alice_change input using key path spending.

In [12]:
# SIGHASH for key path spend
sighash_epoch = bytes.fromhex("00")
hash_type = bytes.fromhex("00")  # SIGHASH_DEFAULT (SIGHASH_ALL)

# Input 1 is the change output from signed_child_tx (locked to alice_change_pubkey).
# We need its outpoint and exact value for the sighash.
change_utxo_value_sat = int(round(decoded_signed_child['vout'][0]['value'] * 100000000))
change_utxo_value_bytes = change_utxo_value_sat.to_bytes(8, byteorder="little", signed=False)

# Transaction data — both inputs must be included in the hashes (SIGHASH_ALL)
sha_prevouts_v2 = sha256(commitment_txid + commitment_index + change_txid + change_index_bytes).digest()
sha_amounts_v2 = sha256(htlc_output_value + change_utxo_value_bytes).digest()

# scriptPubKeys for both inputs
alice_offered_htlc_spk_with_len = varint_len(alice_offered_htlc_spk) + alice_offered_htlc_spk
# Input 1 scriptPubKey: the change output from signed_child_tx uses alice_change_pubkey
alice_change_spk = bytes.fromhex("51") + varint_len(alice_change_pubkey.get_bytes(bip340=True)) + alice_change_pubkey.get_bytes(bip340=True)
alice_change_spk_with_len = varint_len(alice_change_spk) + alice_change_spk
sha_scriptpubkeys_v2 = sha256(alice_offered_htlc_spk_with_len + alice_change_spk_with_len).digest()

sha_sequences_v2 = sha256(sequence_htlc + sequence_change).digest()
sha_outputs_v2 = sha256(outputs_v2).digest()

# Data about this input (key path spend)
spend_type = bytes.fromhex("00")  # key path, no annex

# Index of the input being signed (input 1)
input_index = bytes.fromhex("01000000")

sig_msg_sweeper = (
    sighash_epoch
    + hash_type
    + version
    + locktime
    + sha_prevouts_v2
    + sha_amounts_v2
    + sha_scriptpubkeys_v2
    + sha_sequences_v2
    + sha_outputs_v2
    + spend_type
    + input_index
)

tag_hash = sha256("TapSighash".encode()).digest()
sighash_sweeper = sha256(tag_hash + tag_hash + sig_msg_sweeper).digest()

# Sign with alice_change_privkey (owner of the change UTXO from signed_child_tx)
aux_sweeper = secrets.token_bytes(32)
alice_sweeper_sig = alice_change_privkey.sign_schnorr(sighash_sweeper, aux_sweeper)

# Verify the signature
alice_sweeper_sig_valid = alice_change_pubkey.verify_schnorr(alice_sweeper_sig, sighash_sweeper)
print("Alice change signature valid?", alice_sweeper_sig_valid)


Alice change signature valid? True


### Build the final signed transaction

Now we construct the witness data for both inputs and build the final signed transaction.

In [13]:
# Witness for Input 1 (alice sweeper) - key path spend
witness_input1 = (
    bytes.fromhex("01")  # 1 stack item
    + varint_len(alice_sweeper_sig)
    + alice_sweeper_sig
)

# Complete witness data
witness_v2 = witness + witness_input1

# The final signed transaction
signed_tx_v2 = (
    version
    + marker
    + flag
    + input_count
    + inputs_v2
    + output_count_v2
    + outputs_v2
    + witness_v2
    + locktime
)

print("signed tx v2:", signed_tx_v2.hex())

# Decode the signed transaction
decoded_signed_v2 = node.decoderawtransaction(signed_tx_v2.hex())
print("\n" + json.dumps(decoded_signed_v2, indent=2, default=str))

signed tx v2: 0300000000010249b00259979230f9bd5bc2d4f7a3680d287d2300a3829e878241d0b31ffcc1b802000000000100000086d8d701282b3d9f42cffbfef93cd639dea49a74355a30645cc8741402ae67630000000000ffffffff0220a10700000000002251201bbc6b0949d8e40558a9c958b9b84bb3f9ab632ee9cc2986e6ce6cf0d48f20d860939800000000002251208560a3ef21475305df5fdf049b299f5db8efc55668867ae1212061ee7e901c1c0441212f8c962727d8459d490a94310dd61ad215a66c0f9a8f29ec68869b0cf03bb24228df68388bae823738701b0a682b87718b82394b930382bbe05aed4618304f834192de7253267462a04474c872d60fcb4d3e6ef5bb596dca665792f79d89cb4dae1da7fc1a9e92b4fc5256afc7b813dda786953f210be6f638ffd638db949418b08344207546c6ab3a8c005ce24e9feb7154fd35b62ac0f9125beb5e692d4d912a95de77ad207bd8383df06640c0efe760bb2bac1a1585bdc664e6f1cb61c41a76a6f2be3484ac41c0c39a6b0cffa569f243cec8252a1e5f93b4b072247ff18773917b72fb6341f6082fbd0ee9ad97225e09e436e3e9965d3b1bea9c935aa04af360d50ae3f87402310140261aa8fc24187cb08941ae582cb303772993eaeed744a1e5a9b2252558fba95d4541f4dfa1e902b5d708d9ed8a2041

In [14]:
# Test mempool accept
print("\nTest mempool accept:")
try:
    result = node.testmempoolaccept(rawtxs=[signed_tx_v2.hex()])
    print(json.dumps(result, indent=2, default=str))
    
    if result[0]['allowed']:
        print("\n✓ Transaction is valid and meets minimum relay fee!")
        print(f"  Fee: {result[0].get('fees', {}).get('base', 'N/A')} BTC")
    else:
        print(f"\n✗ Transaction rejected: {result[0].get('reject-reason', 'Unknown')}")
except Exception as e:
    print(f"Error: {e}")


Test mempool accept:
[
  {
    "txid": "9f16971f6a5f723e89a2935a3e7f2fec0dfd2976f271018783e7f2186b6d7166",
    "wtxid": "240f2c5b33d8fba95b6224e7149b5bd00b5ff1392b839a83222c2d24ff83f3bf",
    "allowed": true,
    "vsize": 262,
    "fees": {
      "base": "0.00000300",
      "effective-feerate": "0.00001145",
      "effective-includes": [
        "240f2c5b33d8fba95b6224e7149b5bd00b5ff1392b839a83222c2d24ff83f3bf"
      ]
    }
  }
]

✓ Transaction is valid and meets minimum relay fee!
  Fee: 0.00000300 BTC
